In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
import collections
import heapq

BLUE   = "#3b82f6"
ORANGE = "#f97316"
GREEN  = "#22c55e"
PURPLE = "#8b5cf6"
PINK   = "#ec4899"

In [ ]:
# ── Visualization 1: Random Points ───────────────────────────────────────────
# Human intuition (jittered grid) vs true random, density-colored

N = 250
N_SAMPLES = 5

g = int(np.ceil(np.sqrt(N)))
gx = np.linspace(0.05, 0.95, g)
gy = np.linspace(0.05, 0.95, g)
xx, yy = np.meshgrid(gx, gy)
rng = np.random.RandomState(0)
xh = (xx.ravel()[:N] + rng.normal(0, 0.022, N)).clip(0.01, 0.99)
yh = (yy.ravel()[:N] + rng.normal(0, 0.022, N)).clip(0.01, 0.99)
dh = gaussian_kde(np.vstack([xh, yh]))(np.vstack([xh, yh]))

samples = []
for s in range(N_SAMPLES):
    r = np.random.RandomState(s + 7)
    xr, yr = r.uniform(0, 1, N), r.uniform(0, 1, N)
    dr = gaussian_kde(np.vstack([xr, yr]))(np.vstack([xr, yr]))
    samples.append((xr, yr, dr))

fig1 = make_subplots(1, 2,
    subplot_titles=["What feels random", "What is actually random"],
    horizontal_spacing=0.08)

fig1.add_trace(go.Scatter(
    x=xh, y=yh, mode="markers",
    marker=dict(size=7, color=dh, colorscale="Blues", showscale=False, opacity=0.9),
    hoverinfo="skip",
), row=1, col=1)

for i, (xr, yr, dr) in enumerate(samples):
    fig1.add_trace(go.Scatter(
        x=xr, y=yr, mode="markers",
        visible=(i == 0),
        marker=dict(size=7, color=dr, colorscale="Reds", showscale=False, opacity=0.9),
        hoverinfo="skip",
    ), row=1, col=2)

buttons = [dict(
    label=f"Sample {i+1}",
    method="update",
    args=[{"visible": [True] + [j == i for j in range(N_SAMPLES)]}],
) for i in range(N_SAMPLES)]

fig1.update_layout(
    title=dict(text=f"{N} Points: Human Intuition vs True Random", font=dict(size=16)),
    height=480, template="plotly_white", showlegend=False,
    margin=dict(t=120, l=10, r=10, b=10),
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.72, xanchor="left", y=1.18,
        buttons=buttons, showactive=True,
        bgcolor="white", bordercolor="#e5e7eb",
    )],
    annotations=[dict(
        text="Resample:", x=0.60, xref="paper", y=1.19, yref="paper",
        showarrow=False, font=dict(size=12), xanchor="left",
    )],
    font=dict(family="Inter, Arial, sans-serif"),
)

for c in [1, 2]:
    fig1.update_xaxes(range=[0, 1], showticklabels=False, showgrid=False,
        zeroline=False, showline=True, linecolor="#e5e7eb", mirror=True, row=1, col=c)
    fig1.update_yaxes(range=[0, 1], showticklabels=False, showgrid=False,
        zeroline=False, showline=True, linecolor="#e5e7eb", mirror=True, row=1, col=c)

fig1.write_html("../images/random_points.html", include_plotlyjs="cdn")
fig1.show()

In [ ]:
# ── Visualization 2: Coin Flip Runs ──────────────────────────────────────────
# 10x10 flip grid + run length distribution, toggle true random vs human-like

def human_flips(n, seed=42):
    rng = np.random.RandomState(seed)
    seq = [rng.randint(0, 2)]
    for _ in range(n - 1):
        seq.append(1 - seq[-1] if rng.random() < 0.70 else seq[-1])
    return np.array(seq)

def run_lengths(seq):
    runs, count = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            count += 1
        else:
            runs.append(count)
            count = 1
    runs.append(count)
    return runs

def run_dist(runs, max_len=7):
    d = np.zeros(max_len, dtype=int)
    for r in runs:
        d[min(r, max_len) - 1] += 1
    return d

N = 100
rng = np.random.RandomState(17)
seq_true  = rng.randint(0, 2, N)
seq_human = human_flips(N, seed=5)

runs_true  = run_lengths(seq_true)
runs_human = run_lengths(seq_human)
max_true   = max(runs_true)
max_human  = max(runs_human)

MAX_LEN = 7
dist_true  = run_dist(runs_true,  MAX_LEN)
dist_human = run_dist(runs_human, MAX_LEN)
run_labels = [str(i + 1) if i < MAX_LEN - 1 else f"{MAX_LEN}+" for i in range(MAX_LEN)]

grid_true  = seq_true.reshape(10, 10).tolist()
grid_human = seq_human.reshape(10, 10).tolist()

fig2 = make_subplots(1, 2,
    subplot_titles=["Flip sequence (blue = H, orange = T)", "Run length distribution"],
    column_widths=[0.45, 0.55],
    horizontal_spacing=0.12)

colorscale = [[0, ORANGE], [0.5, ORANGE], [0.5, BLUE], [1.0, BLUE]]

fig2.add_trace(go.Heatmap(
    z=grid_true, colorscale=colorscale, showscale=False, zmin=0, zmax=1,
    xgap=2, ygap=2, visible=True,
    hovertemplate="Row %{y}, Col %{x}: %{z}<extra>True Random</extra>",
), row=1, col=1)

fig2.add_trace(go.Heatmap(
    z=grid_human, colorscale=colorscale, showscale=False, zmin=0, zmax=1,
    xgap=2, ygap=2, visible=False,
    hovertemplate="Row %{y}, Col %{x}: %{z}<extra>Human-like</extra>",
), row=1, col=1)

fig2.add_trace(go.Bar(
    x=dist_true, y=run_labels, orientation="h",
    marker_color=BLUE, opacity=0.85, visible=True,
    hovertemplate="Length %{y}: %{x} runs<extra>True Random</extra>",
), row=1, col=2)

fig2.add_trace(go.Bar(
    x=dist_human, y=run_labels, orientation="h",
    marker_color=ORANGE, opacity=0.85, visible=False,
    hovertemplate="Length %{y}: %{x} runs<extra>Human-like</extra>",
), row=1, col=2)

buttons2 = [
    dict(label="True Random", method="update",
         args=[{"visible": [True, False, True, False]},
               {"title": f"100 Coin Flips: True Random  |  Longest run: {max_true}"}]),
    dict(label="Human-like", method="update",
         args=[{"visible": [False, True, False, True]},
               {"title": f"100 Coin Flips: Human-like  |  Longest run: {max_human}"}]),
]

fig2.update_layout(
    title=dict(text=f"100 Coin Flips: True Random  |  Longest run: {max_true}", font=dict(size=15)),
    height=460, template="plotly_white", showlegend=False,
    margin=dict(t=120, l=10, r=10, b=10), bargap=0.25,
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.0, xanchor="left", y=1.18,
        buttons=buttons2, showactive=True,
        bgcolor="white", bordercolor="#e5e7eb",
    )],
    font=dict(family="Inter, Arial, sans-serif"),
)

fig2.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
fig2.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, autorange="reversed", row=1, col=1)
fig2.update_xaxes(title_text="Number of runs", row=1, col=2)
fig2.update_yaxes(title_text="Run length", row=1, col=2)

fig2.write_html("../images/coin_runs.html", include_plotlyjs="cdn")
fig2.show()

In [ ]:
# ── Visualization 3: Playlist Shuffle ────────────────────────────────────────
# Two-row heatmap: true random vs smart shuffle, colored by artist

def smart_shuffle(songs, rng):
    buckets = collections.defaultdict(list)
    for s in songs:
        buckets[s[0]].append(s)
    for v in buckets.values():
        rng.shuffle(v)
    heap = [(-len(v), k) for k, v in buckets.items()]
    heapq.heapify(heap)
    result, last = [], -1
    while heap:
        skipped = []
        chosen = None
        while heap:
            neg_c, artist = heapq.heappop(heap)
            if artist != last or not heap:
                chosen = (neg_c, artist)
                break
            skipped.append((neg_c, artist))
        for item in skipped:
            heapq.heappush(heap, item)
        if chosen is None:
            break
        neg_c, artist = chosen
        result.append(buckets[artist].pop())
        last = artist
        if buckets[artist]:
            heapq.heappush(heap, (neg_c + 1, artist))
    return result

def count_adjacent(playlist):
    return sum(1 for i in range(len(playlist) - 1) if playlist[i][0] == playlist[i + 1][0])

N_ARTISTS = 5
SONGS_PER = 8
ARTIST_COLORS = [BLUE, ORANGE, GREEN, PURPLE, PINK]
ARTIST_NAMES  = ["Radiohead", "Kendrick Lamar", "Fiona Apple", "LCD Soundsystem", "Bon Iver"]

rng = np.random.RandomState(3)
songs = [(a, s) for a in range(N_ARTISTS) for s in range(SONGS_PER)]
songs_random = songs.copy()
rng.shuffle(songs_random)
songs_smart = smart_shuffle(songs, rng)

adj_random = count_adjacent(songs_random)
adj_smart  = count_adjacent(songs_smart)

artist_colorscale = []
for i, c in enumerate(ARTIST_COLORS):
    artist_colorscale.append([i / N_ARTISTS, c])
    artist_colorscale.append([(i + 1) / N_ARTISTS, c])

z = [
    [s[0] for s in songs_random],
    [s[0] for s in songs_smart],
]
hover_random = [f"Position {i+1}: {ARTIST_NAMES[s[0]]}" for i, s in enumerate(songs_random)]
hover_smart  = [f"Position {i+1}: {ARTIST_NAMES[s[0]]}" for i, s in enumerate(songs_smart)]

fig3 = go.Figure()

fig3.add_trace(go.Heatmap(
    z=z, zmin=0, zmax=N_ARTISTS,
    colorscale=artist_colorscale, showscale=False,
    xgap=2, ygap=6,
    y=["True Random", "Smart Shuffle"],
    customdata=[hover_random, hover_smart],
    hovertemplate="%{customdata}<extra></extra>",
))

for name, color in zip(ARTIST_NAMES, ARTIST_COLORS):
    fig3.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=12, color=color, symbol="square"),
        name=name, showlegend=True,
    ))

fig3.update_layout(
    title=dict(
        text=(
            f"True Random: {adj_random} same-artist plays back-to-back   |   "
            f"Smart Shuffle: {adj_smart} same-artist plays back-to-back"
        ),
        font=dict(size=14),
    ),
    height=280, template="plotly_white",
    margin=dict(t=80, l=10, r=10, b=60),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.35),
    font=dict(family="Inter, Arial, sans-serif"),
)

fig3.update_xaxes(showticklabels=False, showgrid=False, zeroline=False,
    title_text="Song position in queue")
fig3.update_yaxes(showgrid=False, zeroline=False)

fig3.write_html("../images/playlist_shuffle.html", include_plotlyjs="cdn")
fig3.show()